# Phase 2: Legal NER - GraphProp with Adapters and Partial Fine-tuning

## Task
Legal named entity recognition (NER): find entities (names, courts, statutes, dates and so on) in legal judgments. Every word gets a B/I/O tag, and predictions are scored with strict macro F1 (start, end and label must all match).

Model: the pre-trained GraphProp model with a BiGRU and a linear tag layer on top, fine-tuned in two ways: (1) bottleneck adapters after every block with the rest frozen, (2) partial fine-tuning of only the last PARTIAL_K blocks.

## Input
- Phase-2 NER data (training documents with entity spans, test documents); downloaded from Hugging Face if not attached.
- GraphProp checkpoint .pt and its tokenizer .json.

## Flow of the notebook (from input to output)
1. Set the settings and file paths.
2. Find the checkpoint and tokenizer files.
3. Write the model code (arch_graphprop.py), the NER pipeline (ner_core.py), the fine-tuning methods (ner_peft.py) and the experiment runner (ner_run.py).
4. Load the NER documents and make a document-level train / hold-out split.
5. Build the training settings and learning-rate groups.
6. Load the pre-trained model and tokenize the documents (words are lined up with tokens).
7. Set up the two models (adapters, partial fine-tuning).
8. Train with adapters, score on the hold-out, retrain on all documents and predict the test set.
9. Do the same with partial fine-tuning.
10. Show a results table and plots.

## Output
- Submission zips for both runs (names start with adapter_graphprop and partial_graphprop).
- Reports, saved probabilities and summary.png.

In [1]:
# Settings and file paths
AUTO_DOWNLOAD = True
SMOKE = False
SMOKE_CHARS = 12000
SEED = 42
OUT = "/kaggle/working"
N_VAL_DOCS = 14
WIN_TOKENS = 512
STRIDE = 256
DOUBLE_PASS = False
BIDIRECTIONAL = False
GRU_HIDDEN = 256
RARE_DUP = 1
CLASS_POWER = 0.5
EPOCHS = 36
PATIENCE = 8
MICRO_BS = 8
ACCUM = 2
EVAL_FROM = 4
LR_HEAD = 2e-3
WEIGHT_DECAY = 0.01
FULL_RETRAIN = True

ADAPTER_BOTTLENECK = 96
LR_ADAPTER = 1e-3
PARTIAL_K = 3
LR_PARTIAL = 1e-4
RUN_ADAPTER = True
RUN_PARTIAL = True
CKPT = "/kaggle/input/datasets/preetpatelnirma/graphprop-final-pt-p1/legal_graphprop_random_final_random_90_10.pt"
TOK = "/kaggle/input/datasets/preetpatelnirma/tokenizer-graphprop-p1/tokenizer_graphprop.json"

In [2]:
# Find the model checkpoint and tokenizer files
import os

# It takes exts (file extensions) as input, lists files in the input folder with the given extensions and returns a sorted list of file paths.
def _walk(exts):
    out = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for n in dirs + files:
            p = os.path.join(root, n)
            if p.lower().endswith(exts): out.append(p)
    return sorted(set(out))

# It takes label (name to print), explicit (path set by hand), exts (file extensions), keywords (words the file name should contain), require (words that must be in the name), exclude (words to skip) and defaults (default paths) as input, finds the right input file to use and returns a path of the chosen file.
def pick(label, explicit, exts, keywords=(), require=(), exclude=(), defaults=()):
    if explicit: return explicit
    for d in defaults:
        if os.path.exists(d):
            print(f"{label}: {d}"); return d
    c = [p for p in _walk(exts) if all(r in p.lower() for r in require) and not any(x in p.lower() for x in exclude)]
    k = [p for p in c if any(w in p.lower() for w in keywords)] or c
    if len(k) == 1:
        print(f"{label}: {k[0]}"); return k[0]
    seen = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.count(os.sep) <= 5: seen += [os.path.join(root, n) for n in files + dirs][:12]
    raise RuntimeError(f"{label}: found {len(k)} candidate files {k[:8]}.\n"
                       f"Attach the file with 'Add Input' (or set the path in the Settings cell).\n"
                       f"What is currently attached under /kaggle/input: {seen[:40] if seen else 'NOTHING'}")

CKPT = pick("checkpoint", CKPT, (".pt",), keywords=("graph", "prop"), exclude=("last",))
TOK = pick("tokenizer", TOK, (".json",), keywords=("token",), require=("token",), exclude=("fold",))

In [3]:
%%writefile arch_graphprop.py
# GraphProp model code (saved as arch_graphprop.py)
import os, json, zipfile, tempfile, shutil, re as _re

import torch
import torch.nn as nn
import torch.nn.functional as F

class TokenPosEmbed(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size) and max_len (max length) as input, creates the token and position embedding tables and returns nothing.
    def __init__(self, vocab_size, dim, max_len=1024):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, adds token and position embeddings for the input and returns embedding vectors.
    def forward(self, input_ids, pos_offset=0):
        B, T = input_ids.shape
        positions = (torch.arange(T, device=input_ids.device) + pos_offset).unsqueeze(0).expand(B, -1)
        return self.tok(input_ids) + self.pos(positions)

class FeedForward(nn.Module):
    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mult, dim),
            nn.Dropout(dropout),
        )

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return x + self.net(self.norm(x))

class GraphPropLayer(nn.Module):
    # It takes dim (model size), window (window size) and edge_dim (edge size) as input, creates the graph message-passing layer and returns nothing.
    def __init__(self, dim, window=16, edge_dim=64):
        super().__init__()
        self.window = window
        self.norm = nn.LayerNorm(dim)
        self.edge_key = nn.Linear(dim, edge_dim)
        self.edge_mlp = nn.Sequential(nn.Linear(edge_dim * 2, edge_dim), nn.GELU(), nn.Linear(edge_dim, 1))
        self.value_proj = nn.Linear(dim, dim)
        self.rel_emb = nn.Embedding(window, dim)
        self.out_proj = nn.Linear(dim, dim)

    # It takes x (input tensor) as input, passes messages from nearby previous tokens to each token and returns output vectors.
    def forward(self, x):
        B, T, D = x.shape
        W = self.window
        h = self.norm(x)

        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)

        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=x.device).unsqueeze(1) + torch.arange(W, device=x.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=x.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))

        denom = edge_w.sum(-1, keepdim=True).clamp_min(1e-6)
        agg = num / denom
        out = self.out_proj(agg)
        return x + out

    # It takes h (normalised hidden vectors), valid (mask of valid neighbours) and drop_self (skip the token itself) as input, computes the message sums used by the layer and returns (message sum, edge weight sum).
    def _num_den(self, h, valid=None, drop_self=False):
        B, T, D = h.shape
        W = self.window
        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)
        if valid is not None:
            vpad = torch.cat([torch.ones(B, W - 1, dtype=edge_w.dtype, device=h.device),
                              valid.to(edge_w.dtype)], dim=1)
            edge_w = edge_w * vpad.unfold(1, W, 1)
        if drop_self:
            edge_w = torch.cat([edge_w[..., :W - 1], torch.zeros_like(edge_w[..., :1])], dim=-1)
        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=h.device).unsqueeze(1) + torch.arange(W, device=h.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=h.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))
        return num, edge_w.sum(-1, keepdim=True)

    # It takes x (input tensor) and valid (mask of valid neighbours) as input, passes messages from tokens on both sides and returns output vectors.
    def forward_bidirectional(self, x, valid=None):
        h = self.norm(x)
        nc, dc = self._num_den(h, valid)
        vr = valid.flip(1) if valid is not None else None
        nr, dr = self._num_den(h.flip(1), vr, drop_self=True)
        num = nc + nr.flip(1)
        den = (dc + dr.flip(1)).clamp_min(1e-6)
        return x + self.out_proj(num / den)

class LegalGraphPropLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), window (window size), edge_dim (edge size), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full GraphProp language model and returns nothing.
    def __init__(self, vocab_size, dim=512, n_layers=6, window=16, edge_dim=64, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        self.embed = TokenPosEmbed(vocab_size, dim, max_len)
        self.drop = nn.Dropout(dropout)
        layers = []
        for _ in range(n_layers):
            layers.append(GraphPropLayer(dim, window, edge_dim))
            layers.append(FeedForward(dim, dropout=dropout))
        self.blocks = nn.ModuleList(layers)
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:
            self.lm_head.weight = self.embed.tok.weight

        self.apply(self._init_weights)
        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            if isinstance(blk, GraphPropLayer):
                nn.init.normal_(blk.out_proj.weight, mean=0.0, std=0.02 * residual_scale)
            elif isinstance(blk, FeedForward):
                nn.init.normal_(blk.net[-2].weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), pos_offset (position offset), targets (target token ids), last_idx (last token positions), pad_id (padding id) and label_smoothing (label smoothing) as input, runs the model and returns next-token scores.
    def forward(self, input_ids, pos_offset=0, targets=None, last_idx=None, pad_id=None, label_smoothing=0.0):
        x = self.drop(self.embed(input_ids, pos_offset=pos_offset))
        for blk in self.blocks:
            x = blk(x)
        h = self.norm_f(x)
        if last_idx is not None:
            rows = torch.arange(h.size(0), device=h.device)
            return self.lm_head(h[rows, last_idx])
        logits = self.lm_head(h)
        if targets is None:
            return logits
        loss_sum = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=pad_id, reduction="sum", label_smoothing=label_smoothing)
        n_tok = (targets != pad_id).sum()
        return loss_sum.reshape(1), n_tok.reshape(1)

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}
_PRETOK_RE = _re.compile(r"\'s|\'t|\'re|\'ve|\'m|\'ll|\'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]
        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None
        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]
        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

class Tok:

    # It takes inner (inner tokenizer) as input, loads the tokenizer and returns nothing.
    def __init__(self, inner):
        self.inner = inner
        self.vocab_size = len(inner)
        self.pad_id = inner.pad_token_id
        self.eos_id = inner.eos_token_id
        self.bos_id = None

    # It takes text as input, converts text into token ids and returns a list of token ids.
    def encode(self, text):
        return self.inner.encode(text)

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        return self.inner.decode(ids, skip_special_tokens)

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):
        inner = self.inner
        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            s = m.group()
            b = s.encode("utf-8")
            byte2char = []
            for k, ch in enumerate(s):
                byte2char.extend([m.start() + k] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[x] for x in b)
            pos = 0
            for piece in inner._bpe(sym):
                n = len(piece)
                ids.append(inner.vocab[piece])
                offs.append((byte2char[pos], byte2char[pos + n - 1] + 1))
                pos += n
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):
    return Tok(LegalByteBPETokenizer.load(tok_path))

# It takes path (file path) as input, turns an unzipped checkpoint folder back into a .pt file and returns (file path, temporary folder or None).
def _materialise_checkpoint_path(path):
    if not os.path.isdir(path):
        return path, None
    pkl = None
    for root, _d, files in os.walk(path):
        if "data.pkl" in files:
            pkl = root
            break
    if pkl is None:
        raise FileNotFoundError(f"{path} is a directory but contains no data.pkl")
    tmpdir = tempfile.mkdtemp(prefix="gp_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    top = "archive"
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as zf:
        for root, _d, files in os.walk(pkl):
            for fn in sorted(files):
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, pkl).replace(os.sep, "/")
                zf.write(full, arcname=f"{top}/{rel}")
    return out, tmpdir

# It takes obj (object) as input, gets the weights dictionary out of a saved checkpoint and returns weights dictionary.
def _extract_state_dict(obj):
    if isinstance(obj, nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        for k in ("best_state", "state_dict", "model_state_dict", "model_state", "model", "weights"):
            v = obj.get(k)
            if isinstance(v, nn.Module):
                return v.state_dict()
            if isinstance(v, dict) and any(torch.is_tensor(t) for t in v.values()):
                return v
        if any(torch.is_tensor(t) for t in obj.values()):
            return obj
    raise ValueError("could not find a state_dict in checkpoint")

# It takes sd (weights dictionary) as input, removes extra name prefixes from weight names and returns weights dictionary with clean names.
def _strip_prefixes(sd):
    out = {}
    for k, v in sd.items():
        while k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out

# It takes sd (weights dictionary) as input, works out the model settings from the saved weights and returns a dictionary of model settings.
def infer_config(sd):
    vocab, dim = sd["embed.tok.weight"].shape
    max_len = sd["embed.pos.weight"].shape[0]
    idx = {int(m.group(1)) for k in sd for m in [_re.match(r"blocks\.(\d+)\.", k)] if m}
    n_blocks = max(idx) + 1
    assert n_blocks % 2 == 0, "blocks should alternate GraphPropLayer/FeedForward"
    window = sd["blocks.0.rel_emb.weight"].shape[0]
    edge_dim = sd["blocks.0.edge_key.weight"].shape[0]
    return dict(vocab_size=vocab, dim=dim, n_layers=n_blocks // 2, window=window,
                edge_dim=edge_dim, max_len=max_len)

class Backbone:

    causal = True

    # It takes model and cfg (settings) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg):
        self.model = model
        self.cfg = cfg
        self.d_model = cfg["dim"]
        self.max_len = cfg["max_len"]
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag=True):
        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional

    # It takes input_ids (token ids), attention_mask (mask of real tokens) and pos_offset (position offset) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None, pos_offset=0):
        m = self.model
        T = input_ids.size(1)
        assert T + pos_offset <= self.max_len, f"sequence {T}+{pos_offset} exceeds max_len={self.max_len}"
        x = m.drop(m.embed(input_ids, pos_offset=pos_offset))
        if self.bidirectional:
            valid = attention_mask.bool() if attention_mask is not None else None
            for blk in m.blocks:
                x = blk.forward_bidirectional(x, valid) if isinstance(blk, GraphPropLayer) else blk(x)
        else:
            for blk in m.blocks:
                x = blk(x)
        return m.norm_f(x)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, returns next-token scores.
    def logits(self, input_ids, pos_offset=0):
        return self.model(input_ids, pos_offset=pos_offset)

    # It returns the list of model blocks.
    def block_modules(self):
        return list(self.model.blocks)

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):
        return [f"blocks.{n}" if n else "blocks"
                for n, mod in self.model.blocks.named_modules() if isinstance(mod, nn.Linear)]

# It takes ckpt_path (checkpoint path), device and dropout (dropout rate) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", dropout=0.1):
    path, tmp = _materialise_checkpoint_path(ckpt_path)
    try:
        obj = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    sd = _strip_prefixes(_extract_state_dict(obj))
    cfg = infer_config(sd)
    tied = ("lm_head.weight" not in sd) or torch.equal(sd["lm_head.weight"], sd["embed.tok.weight"])
    model = LegalGraphPropLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                             window=cfg["window"], edge_dim=cfg["edge_dim"], max_len=cfg["max_len"],
                             dropout=dropout, tie=tied)
    if tied and "lm_head.weight" not in sd:
        sd = dict(sd); sd["lm_head.weight"] = sd["embed.tok.weight"]
    res = model.load_state_dict(sd, strict=False)
    if res.missing_keys or res.unexpected_keys:
        print("!" * 70)
        print("!!! STATE-DICT MISMATCH when loading GraphProp checkpoint !!!")
        print("missing:", res.missing_keys)
        print("unexpected:", res.unexpected_keys)
        print("!" * 70)
        raise RuntimeError("GraphProp checkpoint keys do not match model")
    model.to(device).eval()
    cfg["tied"] = tied
    return Backbone(model, cfg)

Writing arch_graphprop.py


In [4]:
%%writefile ner_core.py
# Main NER code: data, labels, training, scoring (saved as ner_core.py)
import os, re, json, glob, math, time, random, zipfile, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
T2I = {t: i for i, t in enumerate(TAGS)}
NT = len(TAGS)
RARE = ("WIT", "A.COUNSEL", "R.COUNSEL", "APP", "RESP")
WORD_RE = re.compile(r"[^\W\d_]+|\d+|[^\w\s]|_")

# It takes name and roots (folders to search) as input, searches the input folders for a file and returns a file path (or None if not found).
def find_file(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    for r in roots:
        hits = glob.glob(os.path.join(r, "**", name), recursive=True)
        hits = [h for h in hits if "__MACOSX" not in h]
        if hits:
            return sorted(hits, key=len)[0]
    return None

# It loads the training documents and their entity spans and returns a dictionary of documents with text and spans.
def load_train_docs():
    docs = {}
    for f in (1, 2):
        tp, gp = find_file(f"fold{f}-text.json"), find_file(f"fold{f}-gold.json")
        assert tp and gp, (f"fold{f}-text.json / fold{f}-gold.json not found under /kaggle/input - "
                           "attach the dataset that contains the Phase-2 Train files")
        T, G = json.load(open(tp)), json.load(open(gp))
        assert set(T) == set(G), f"fold{f}: text ids and gold ids differ"
        for k in T:
            assert k not in docs, f"duplicate doc id {k} across folds"
            sp = [(int(a), int(b), l) for a, b, l, *_ in G[k]]
            for s, e, l in sp:
                assert l in LABELS and 0 <= s < e <= len(T[k]), (k, s, e, l)
            docs[k] = {"text": T[k], "spans": sorted(sp), "fold": f}
    return docs

# It loads the test documents and returns a dictionary of test documents.
def load_test_docs():
    p = find_file("fold3-text.json")
    if not p:
        return None
    T = json.load(open(p))
    return {k: {"text": v, "spans": []} for k, v in T.items()}

# It takes docs (documents), n_val (number of hold-out documents), seed (random seed) and tries (number of random tries) as input, splits documents into train and hold-out, keeping rare labels in both and returns (train ids, hold-out ids).
def make_split(docs, n_val=14, seed=0, tries=4000):
    ids = sorted(docs)
    cnt = {k: np.array([sum(1 for s in docs[k]["spans"] if s[2] == l) for l in LABELS]) for k in ids}
    rng = random.Random(seed)
    best, bscore = None, -1
    tot = sum(cnt.values())
    for _ in range(tries):
        v = rng.sample(ids, n_val)
        c = sum(cnt[k] for k in v)
        score = (c / np.maximum(tot, 1)).min() - 0.5 * abs((c / np.maximum(tot, 1)).mean() - n_val / len(ids))
        if score > bscore:
            best, bscore = v, score
    val = sorted(best)
    return [k for k in ids if k not in set(val)], val

# It takes text as input, splits text into small words and remember their positions and returns (word start positions, word end positions).
def segment(text):
    m = [(x.start(), x.end()) for x in WORD_RE.finditer(text)]
    ws = np.array([a for a, _ in m], dtype=np.int64)
    we = np.array([b for _, b in m], dtype=np.int64)
    return ws, we

# It takes ws (word start positions), we (word end positions) and spans (entity spans) as input, gives each word its B/I/O tag and returns a list of tag ids.
def tag_words(ws, we, spans):
    tags = np.zeros(len(ws), dtype=np.int64)
    for s, e, l in spans:
        i0 = int(np.searchsorted(we, s, side="right"))
        i1 = int(np.searchsorted(ws, e, side="left"))
        if i1 <= i0:
            continue
        tags[i0] = T2I["B-" + l]
        tags[i0 + 1:i1] = T2I["I-" + l]
    return tags

# It takes tag_ids (tag ids), ws (word start positions) and we (word end positions) as input, turns word tags back into entity spans and returns a list of (start, end, label).
def spans_from_tags(tag_ids, ws, we):
    out, cur = [], None
    for i, t in enumerate(tag_ids):
        tg = TAGS[t]
        if tg == "O":
            if cur: out.append(cur); cur = None
        elif tg[0] == "B":
            if cur: out.append(cur)
            cur = [int(ws[i]), int(we[i]), tg[2:]]
        else:
            if cur and cur[2] == tg[2:]:
                cur[1] = int(we[i])
            else:
                if cur: out.append(cur)
                cur = [int(ws[i]), int(we[i]), tg[2:]]
    if cur: out.append(cur)
    return [tuple(x) for x in out]

_TRANS = None

# It builds the table of allowed tag-to-tag moves and returns (allowed move table, allowed start tags).
def _trans():
    global _TRANS
    if _TRANS is None:
        m = np.zeros((NT, NT), dtype=np.float32)
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-"):
                for a, ta in enumerate(TAGS):
                    if ta[2:] != tb[2:] or ta == "O":
                        m[a, b] = -1e4
        start = np.array([-1e4 if t.startswith("I-") else 0.0 for t in TAGS], dtype=np.float32)
        _TRANS = (m, start)
    return _TRANS

# It takes logp (log probabilities) as input, finds the best tag sequence that follows the B/I/O rules and returns the best label id for each position.
def viterbi(logp):
    n = logp.shape[0]
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    tr, start = _trans()
    score = logp[0] + start
    back = np.zeros((n, NT), dtype=np.int64)
    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

# It takes logp (log probabilities), ws (word start positions), we (word end positions) and mode as input, turns predicted tags into entity spans and returns a list of entity spans.
def decode(logp, ws, we, mode="viterbi"):
    path = viterbi(logp) if mode == "viterbi" else logp.argmax(1)
    return spans_from_tags(path, ws, we)

# It takes gold (true labels) and pred (predictions) as input, computes strict precision, recall and F1 for entity spans and returns a dictionary of scores (overall and per label).
def strict_scores(gold, pred):
    tp = {l: 0 for l in LABELS}; fp = dict(tp); fn = dict(tp)
    for d in gold:
        g = {(int(a), int(b), l) for a, b, l, *_ in gold[d]}
        p = {(int(a), int(b), l) for a, b, l, *_ in pred.get(d, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in LABELS:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = dict(P=P, R=R, F1=(2 * P * R / (P + R) if P + R else 0.0), tp=tp[l], fp=fp[l], fn=fn[l])
    out = {"per_label": per,
           "macro_P": float(np.mean([per[l]["P"] for l in LABELS])),
           "macro_R": float(np.mean([per[l]["R"] for l in LABELS])),
           "macro_F1": float(np.mean([per[l]["F1"] for l in LABELS]))}
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mp, mr = T / max(1, T + FP), T / max(1, T + FN)
    out["micro_F1"] = 2 * mp * mr / max(1e-9, mp + mr)
    return out

# It takes sc (scores) and title as input, prints the scores for each label and returns nothing.
def print_scores(sc, title=""):
    print(title, "strict macro-F1 %.4f  (P %.4f  R %.4f)  micro-F1 %.4f" % (sc["macro_F1"], sc["macro_P"], sc["macro_R"], sc["micro_F1"]))
    for l in LABELS:
        r = sc["per_label"][l]
        print("   %-10s F1 %.3f  P %.3f  R %.3f   tp %4d fp %4d fn %4d" % (l, r["F1"], r["P"], r["R"], r["tp"], r["fp"], r["fn"]))

# It takes docs (documents) and ids (document ids) as input, checks the best score possible with this word splitting and returns scores of the rebuilt gold spans.
def oracle_check(docs, ids):
    pred, gold = {}, {}
    for k in ids:
        ws, we = segment(docs[k]["text"])
        pred[k] = spans_from_tags(tag_words(ws, we, docs[k]["spans"]), ws, we)
        gold[k] = docs[k]["spans"]
    return strict_scores(gold, pred)

class Enc:

    # It takes doc_id (document id), text, tok (tokenizer), spans (entity spans) and soft (teacher probabilities) as input, tokenizes a document and line up its words and tags and returns nothing.
    def __init__(self, doc_id, text, tok, spans=None, soft=None):
        self.id, self.text, self.spans = doc_id, text, spans or []
        ids, offs = tok.encode_with_offsets(text)
        self.ids = np.asarray(ids, dtype=np.int64)
        offs = np.asarray(offs, dtype=np.int64).reshape(-1, 2)
        self.ws, self.we = segment(text)
        starts, ends = offs[:, 0], offs[:, 1]
        n = len(self.ids)
        self.wl = np.clip(np.searchsorted(starts, self.we, side="left") - 1, 0, n - 1)
        self.wf = np.clip(np.searchsorted(ends, self.ws, side="right"), 0, n - 1)
        self.wf = np.minimum(self.wf, self.wl)
        self.tags = tag_words(self.ws, self.we, self.spans) if spans is not None else None
        self.soft = soft

    # It returns the number of words.
    @property
    def n_words(self):
        return len(self.ws)

# It takes docs (documents), ids (document ids) and tok (tokenizer) as input, tokenizes all documents and returns a list of encoded documents.
def build_encs(docs, ids, tok):
    t0 = time.time()
    out = [Enc(k, docs[k]["text"], tok, docs[k].get("spans", [])) for k in ids]
    print(f"encoded {len(out)} docs, {sum(len(e.ids) for e in out):,} tokens, {sum(e.n_words for e in out):,} words in {time.time()-t0:.0f}s")
    return out

# It takes encs (encoded documents) as input, shows simple token length statistics and returns a dictionary with tokens per character.
def token_stats(encs):
    r = np.mean([len(e.ids) / max(1, len(e.text)) for e in encs])
    return dict(tokens_per_char=float(r), chars_per_token=float(1 / r))

# It takes enc (encoded document), W (window size), stride (window stride) and shift (window shift) as input, cuts a document into overlapping token windows and returns a list of (start, end) windows.
def windows_for(enc, W, stride, shift=0):
    n = len(enc.ids)
    if n <= W:
        return [(0, n)]
    w, a = [], shift % stride if shift else 0
    if a: w.append((0, min(W, n)))
    while True:
        b = min(a + W, n)
        w.append((a, b))
        if b >= n: break
        a += stride
    return w

# It takes enc (encoded document), a (window start) and b (window end) as input, finds the words that lie fully inside a window and returns indices of the words in the window.
def window_words(enc, a, b):
    m = (enc.wf >= a) & (enc.wl < b)
    return np.nonzero(m)[0]

CTX_TOKENS = 0

# It takes items (list of items), double (read text twice), sep_id (separator id), pad_id (padding id) and device as input, builds padded tensors for one batch and returns a dictionary of tensors.
def make_batch(items, double, sep_id, pad_id, device):
    H = CTX_TOKENS
    assert not (H and double), "header context and double-pass cannot be combined"
    L = max((b - a) for _, a, b, _ in items) + (H + 1 if H else 0)
    TL = (2 * L + 1) if double else L
    Wm = max(1, max(len(wi) for *_, wi in items))
    B = len(items)
    ids = np.full((B, TL), pad_id, dtype=np.int64)
    widx = np.zeros((B, Wm), dtype=np.int64)
    wmask = np.zeros((B, Wm), dtype=bool)
    tags = np.full((B, Wm), -100, dtype=np.int64)
    soft = None
    if any(e.soft is not None for e, *_ in items):
        soft = np.zeros((B, Wm, NT), dtype=np.float32)
    for i, (e, a, b, wi) in enumerate(items):
        off = 0
        if H and a > 0:
            if a <= H:
                seg = e.ids[:b]
                n = len(seg); ids[i, :n] = seg
                k = len(wi); widx[i, :k] = e.wl[wi]; wmask[i, :k] = True
                if e.tags is not None: tags[i, :k] = e.tags[wi]
                if soft is not None and e.soft is not None: soft[i, :k] = e.soft[wi]
                continue
            seg = np.concatenate([e.ids[:H], [sep_id], e.ids[a:b]]); n = len(seg)
            ids[i, :n] = seg
            off = H + 1
        else:
            seg = e.ids[a:b]; n = len(seg)
            ids[i, :n] = seg
        if double:
            ids[i, n] = sep_id
            ids[i, n + 1:2 * n + 1] = seg
            off = n + 1
        k = len(wi)
        widx[i, :k] = e.wl[wi] - a + off
        wmask[i, :k] = True
        if e.tags is not None: tags[i, :k] = e.tags[wi]
        if soft is not None and e.soft is not None: soft[i, :k] = e.soft[wi]
    t = lambda x: torch.from_numpy(x).to(device)
    out = dict(ids=t(ids), widx=t(widx), wmask=t(wmask), tags=t(tags))
    if soft is not None: out["soft"] = t(soft)
    return out

# It takes encs (encoded documents), W (window size), stride (window stride), epoch_seed (seed for this epoch), rare_dup (copies of rare windows) and min_words (minimum words) as input, makes the list of training windows for one epoch and returns a list of training windows.
def train_items(encs, W, stride, epoch_seed, rare_dup=1, min_words=1):
    rng = random.Random(epoch_seed)
    items = []
    rare_ids = {T2I["B-" + l] for l in RARE}
    for e in encs:
        for a, b in windows_for(e, W, stride, shift=rng.randrange(stride)):
            wi = window_words(e, a, b)
            if len(wi) < min_words: continue
            items.append((e, a, b, wi))
            if rare_dup and e.tags is not None and any(int(t) in rare_ids for t in e.tags[wi]):
                items.extend([(e, a, b, wi)] * rare_dup)
    rng.shuffle(items)
    return items

class NERModel(nn.Module):

    # It takes bb (pre-trained backbone), gru_hidden (GRU size), dropout (dropout rate), pool (pooling type), double (read text twice), use_gru (use a GRU) and sep_id (separator id) as input, builds the tagging model: backbone plus BiGRU and tag layer and returns nothing.
    def __init__(self, bb, gru_hidden=256, dropout=0.2, pool="last", double=False, use_gru=True, sep_id=1):
        super().__init__()
        self.bb_model = bb.model
        self.bb = bb
        self.pool, self.double, self.use_gru, self.sep_id = pool, double, use_gru, sep_id
        d = bb.d_model
        self.drop = nn.Dropout(dropout)
        if use_gru:
            self.gru = nn.GRU(d, gru_hidden, batch_first=True, bidirectional=True)
            dout = 2 * gru_hidden
        else:
            dout = d
        self.out = nn.Linear(dout, NT)

    # It takes ids (token ids), widx (token index of each word) and wmask (word mask) as input, runs the model and returns tag scores for each word.
    def forward(self, ids, widx, wmask):
        shim = copy.copy(self.bb); shim.model = self.bb_model
        h = shim.hidden(ids)
        idx = widx.unsqueeze(-1).expand(-1, -1, h.size(-1))
        x = torch.gather(h, 1, idx)
        x = self.drop(x.float())
        if self.use_gru:
            lens = wmask.sum(1).clamp(min=1).cpu()
            pk = nn.utils.rnn.pack_padded_sequence(x, lens, batch_first=True, enforce_sorted=False)
            y, _ = self.gru(pk)
            x, _ = nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=widx.size(1))
            x = self.drop(x)
        return self.out(x)

# It takes model as input, counts trainable and total parameters and returns (trainable parameters, total parameters).
def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    al = sum(p.numel() for p in model.parameters())
    return tr, al

# It takes e (error) as input, checks whether an error is a GPU out-of-memory error and returns True or False.
def _is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()

# It takes model, encs (encoded documents), W (window size), stride (window stride), bs (batch size), double (read text twice), pad_id (padding id), sep_id (separator id), device and use_amp (use mixed precision) as input, predicts probabilities, averaging overlapping windows and returns probabilities for every document.
@torch.no_grad()
def predict_probs(model, encs, W, stride, bs=8, double=False, pad_id=0, sep_id=1, device="cuda", use_amp=True):
    model.eval()
    outs = [np.zeros((e.n_words, NT), dtype=np.float32) for e in encs]
    wsum = [np.zeros(e.n_words, dtype=np.float32) for e in encs]
    items = []
    for di, e in enumerate(encs):
        for a, b in windows_for(e, W, stride):
            wi = window_words(e, a, b)
            if len(wi): items.append((di, e, a, b, wi))
    items.sort(key=lambda x: len(x[1].ids[x[2]:x[3]]))
    i, cur = 0, bs
    while i < len(items):
        chunk = items[i:i + cur]
        batch = make_batch([(e, a, b, wi) for _, e, a, b, wi in chunk], double, sep_id, pad_id, device)
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp and device != "cpu"):
                lg = model(batch["ids"], batch["widx"], batch["wmask"])
        except RuntimeError as ex:
            if _is_oom(ex) and cur > 1:
                torch.cuda.empty_cache(); cur = max(1, cur // 2); print(f"  OOM in inference -> batch {cur}"); continue
            raise
        pr = torch.softmax(lg.float(), -1).cpu().numpy()
        for j, (di, e, a, b, wi) in enumerate(chunk):
            k = len(wi)
            pos = np.arange(k)
            w = (np.minimum(pos, k - 1 - pos) + 1).astype(np.float32)
            outs[di][wi] += pr[j, :k] * w[:, None]
            wsum[di][wi] += w
        i += len(chunk)
    for o, s in zip(outs, wsum):
        miss = s == 0
        o[~miss] /= s[~miss][:, None]
        if miss.any(): o[miss, 0] = 1.0
    return outs

# It takes encs (encoded documents), probs (probabilities) and mode as input, turns tag probabilities into entity spans and returns predicted spans per document.
def probs_to_spans(encs, probs, mode="viterbi"):
    return {e.id: decode(np.log(np.clip(p, 1e-9, 1)), e.ws, e.we, mode) for e, p in zip(encs, probs)}

# It takes model, encs (encoded documents), W (window size), stride (window stride), bs (batch size), double (read text twice), pad_id (padding id), sep_id (separator id), device and mode as input, predicts on documents and computes the scores and returns (scores, probabilities, predictions).
def evaluate(model, encs, W, stride, bs=8, double=False, pad_id=0, sep_id=1, device="cuda", mode="viterbi"):
    probs = predict_probs(model, encs, W, stride, bs, double, pad_id, sep_id, device)
    pred = probs_to_spans(encs, probs, mode)
    gold = {e.id: e.spans for e in encs}
    return strict_scores(gold, pred), probs, pred

# It takes encs (encoded documents), power and cap (maximum value) as input, gives rare tags a higher loss weight and returns a tensor of tag weights.
def tag_class_weights(encs, power=0.5, cap=6.0):
    cnt = np.ones(NT)
    for e in encs:
        cnt += np.bincount(e.tags[e.tags >= 0], minlength=NT)
    ent = cnt[1:]
    w = np.ones(NT)
    ref = np.median(ent)
    w[1:] = np.clip((ref / ent) ** power, 0.5, cap)
    return torch.tensor(w, dtype=torch.float32)

# It takes model as input, copies the trainable weights and returns a dictionary of trainable weights.
def _tstate(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if _trainable_key(model, k)}

PROGRESS = dict(total=0.0, done=0.0, skipped=0.0, t0=None)

# It takes sec (seconds) as input, formats seconds as hours:minutes:seconds and returns time as text.
def _hms(sec):
    sec = int(max(0, sec)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

# It shows overall progress and time left and returns progress text.
def progress_line():
    P = PROGRESS
    if not P["total"] or not P["t0"] or P["done"] <= 0: return ""
    el = time.time() - P["t0"]; rate = P["done"] / max(1e-9, el); left = max(0.0, P["total"] - P["done"] - P["skipped"])
    return f"[overall {100 * min(1.0, (P['done'] + P['skipped']) / P['total']):.0f}% | elapsed {_hms(el)} | ETA {_hms(left / rate)}]"

# It takes model, train_encs (encoded training documents), val_encs (encoded validation documents), cfg (settings), param_groups (parameter groups), device and log (print function) as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(model, train_encs, val_encs, cfg, param_groups, device="cuda", log=print):
    use_amp = cfg.get("amp", True) and device != "cpu"
    opt = torch.optim.AdamW(param_groups, betas=(0.9, 0.98), weight_decay=cfg.get("wd", 0.01))
    base_lr = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw = tag_class_weights(train_encs, cfg.get("class_power", 0.5)).to(device)
    dmodel = nn.DataParallel(model) if (device == "cuda" and torch.cuda.device_count() > 1) else model
    hint = cfg.get("bs_state")
    bs, accum = (min(cfg["bs"], hint["bs"]) if hint else cfg["bs"]), cfg["accum"]
    eff = cfg["bs"] * accum
    n_items = len(train_items(train_encs, cfg["W"], cfg["stride"], 0, cfg.get("rare_dup", 1)))
    steps_per_epoch = max(1, n_items // eff)
    total = steps_per_epoch * cfg["epochs"]
    warm = max(1, int(cfg.get("warmup", 0.06) * total))
    log(f"windows/epoch {n_items}, optimizer steps/epoch {steps_per_epoch}, total {total}, micro-batch {bs} x accum {accum}")
    step, best, best_ep, bad, hist, best_state = 0, -1.0, -1, 0, [], None
    avg_state, avg_n, ep0 = None, 0, 0
    kd_a, kd_T = cfg.get("kd_alpha", 0.0), cfg.get("kd_T", 2.0)
    train_params = [p for g in param_groups for p in g["params"]]
    patience = cfg.get("patience", 4)
    swa_last, snaps = int(cfg.get("swa_last", 0)), set(cfg.get("snapshot_epochs") or ())
    ckpt, save_every = cfg.get("ckpt_path"), float(cfg.get("save_every", 1800))
    if ckpt and os.path.exists(ckpt):
        try:
            ck = torch.load(ckpt, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["state"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
            step, ep0, best, best_ep, bad, hist = ck["step"], ck["ep"], ck["best"], ck["best_ep"], ck["bad"], ck["hist"]
            best_state, avg_state, avg_n = ck["best_state"], ck["avg_state"], ck["avg_n"]
            log(f"  resumed from {os.path.basename(ckpt)} at epoch {ep0 + 1}/{cfg['epochs']}"); del ck
            PROGRESS["skipped"] += ep0 * len(train_encs)
        except Exception as ex:
            log(f"  could not resume from {ckpt} ({str(ex)[:100]}) -> training from scratch"); step, ep0, best, best_ep, bad, hist, best_state, avg_state, avg_n = 0, 0, -1.0, -1, 0, [], None, None, 0
    t_save = time.time()
    for ep in range(ep0, cfg["epochs"]):
        model.train()
        items = train_items(train_encs, cfg["W"], cfg["stride"], cfg.get("seed", 0) * 1000 + ep, cfg.get("rare_dup", 1))
        t0, run, nrun = time.time(), torch.zeros((), device=device), 0
        if PROGRESS["t0"] is None: PROGRESS["t0"] = t0
        try:
            from tqdm.auto import tqdm as _tqdm
            bar = _tqdm(total=steps_per_epoch, desc=f"epoch {ep + 1}/{cfg['epochs']}", mininterval=20, leave=False, dynamic_ncols=True)
        except Exception:
            bar = None
        i = 0
        while i + eff <= len(items) or (i == 0 and len(items)):
            lr_f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            lr_f = max(lr_f, 0.02) if step >= warm else lr_f
            for g, b0 in zip(opt.param_groups, base_lr): g["lr"] = b0 * lr_f
            micro = items[i:i + eff]; i += eff
            j = 0
            while j < len(micro):
                part = micro[j:j + bs]
                batch = make_batch(part, cfg.get("double", False), cfg.get("sep_id", 1), cfg.get("pad_id", 0), device)
                try:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                        lg = dmodel(batch["ids"], batch["widx"], batch["wmask"]).float()
                    mask = batch["tags"] != -100
                    ce = F.cross_entropy(lg[mask], batch["tags"][mask], weight=cw, label_smoothing=cfg.get("label_smooth", 0.0))
                    loss = ce
                    if kd_a > 0 and "soft" in batch:
                        lp = F.log_softmax(lg[mask] / kd_T, -1)
                        tgt = batch["soft"][mask]
                        has = tgt.sum(-1) > 0
                        if bool(has.any()):
                            tgt = tgt[has].clamp_min(1e-9) ** (1 / kd_T); tgt = tgt / tgt.sum(-1, keepdim=True)
                            kd = F.kl_div(lp[has], tgt, reduction="batchmean") * kd_T * kd_T
                            frac = has.float().mean()
                            loss = (1 - kd_a * frac) * ce + kd_a * frac * kd
                    scaler.scale(loss * len(part) / len(micro)).backward()
                    run += ce.detach().float(); nrun += 1
                    j += len(part)
                except RuntimeError as ex:
                    if dmodel is not model and not _is_oom(ex):
                        log(f"  DataParallel failed ({str(ex)[:120]}) -> continuing on a single GPU")
                        dmodel = model; opt.zero_grad(set_to_none=True); j = 0; continue
                    if _is_oom(ex) and bs > 1:
                        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                        bs = max(1, bs // 2)
                        if hint is not None: hint["bs"] = bs
                        log(f"  OOM -> micro-batch {bs}"); j = 0; continue
                    raise
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(train_params, cfg.get("clip", 1.0))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if bar is not None:
                bar.update(1)
                if step % 25 == 0: bar.set_postfix(ce=f"{float(run) / max(1, nrun):.3f}")
        if bar is not None: bar.close()
        PROGRESS["done"] += len(train_encs)
        rec = dict(epoch=ep + 1, train_ce=float(run) / max(1, nrun), secs=time.time() - t0)
        if val_encs and ep + 1 >= cfg.get("eval_from", 1):
            sc, _, _ = evaluate(model, val_encs, cfg["W"], cfg["stride"], bs=max(2, bs * 2), double=cfg.get("double", False),
                                pad_id=cfg.get("pad_id", 0), sep_id=cfg.get("sep_id", 1), device=device)
            rec.update(val_macro_F1=sc["macro_F1"], val_micro_F1=sc["micro_F1"])
            if sc["macro_F1"] > best:
                best, best_ep, bad = sc["macro_F1"], ep + 1, 0
                best_state = _tstate(model)
            else:
                bad += 1
        if swa_last and ep + 1 > cfg["epochs"] - swa_last:
            cur = _tstate(model)
            if avg_state is None: avg_state, avg_n = {k: v.float() for k, v in cur.items()}, 1
            else:
                avg_n += 1
                for k, v in cur.items(): avg_state[k] += (v.float() - avg_state[k]) / avg_n
        hist.append(rec)
        eta_run = (time.time() - t0) * (cfg["epochs"] - ep - 1)
        log("  ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()) + f"  | this run: epoch {ep + 1}/{cfg['epochs']}, ETA {_hms(eta_run)}  " + progress_line())
        if (ep + 1) in snaps and cfg.get("snapshot_fn"):
            cfg["snapshot_fn"](ep + 1, model)
        if ckpt and ep + 1 < cfg["epochs"] and time.time() - t_save >= save_every:
            torch.save(dict(state=_tstate(model), opt=opt.state_dict(), scaler=scaler.state_dict(), step=step, ep=ep + 1, best=best, best_ep=best_ep,
                            bad=bad, hist=hist, best_state=best_state, avg_state=avg_state, avg_n=avg_n), ckpt + ".tmp")
            os.replace(ckpt + ".tmp", ckpt); t_save = time.time(); log(f"  [saved resumable checkpoint after epoch {ep + 1}]")
        if val_encs and cfg.get("select", "best") == "best" and patience > 0 and bad >= patience:
            log(f"early stop (no improvement for {bad} evaluations)"); break
    last_state = _tstate(model)
    if not val_encs or cfg.get("select", "best") == "last" or best_state is None:
        best_state, best_ep = last_state, len(hist)
    if cfg.get("return_last"):
        avg_out = {k: v.to(last_state[k].dtype) if last_state[k].is_floating_point() else last_state[k] for k, v in avg_state.items()} if avg_state else None
        return hist, best_ep, best_state, {"last": last_state, "avg": avg_out}
    return hist, best_ep, best_state

# It takes model and k (weight name) as input, checks whether a weight is trainable and returns True or False.
def _trainable_key(model, k):
    sd_params = dict(model.named_parameters())
    return (k in sd_params and sd_params[k].requires_grad)

# It takes model and state (weights) as input, loads the best saved weights into the model and returns nothing.
def load_best(model, state):
    miss = model.load_state_dict(state, strict=False)
    assert not miss.unexpected_keys, miss.unexpected_keys

# It takes pred (predictions), test_docs (test documents), path_zip (zip path) and with_text (include text) as input, checks predictions and writes the submission zip and returns a path of the zip.
def write_submission(pred, test_docs, path_zip, with_text=False):
    assert set(pred) == set(test_docs), f"id mismatch: missing {len(set(test_docs)-set(pred))}, extra {len(set(pred)-set(test_docs))}"
    out = {}
    for k, sp in pred.items():
        n = len(test_docs[k]["text"]); rows = []
        for s, e, l in sorted(sp):
            assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= n and l in LABELS, (k, s, e, l)
            rows.append([s, e, l, test_docs[k]["text"][s:e]] if with_text else [s, e, l])
        out[k] = rows
    os.makedirs(os.path.dirname(path_zip) or ".", exist_ok=True)
    with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(out))
    n_ent = sum(len(v) for v in out.values())
    print(f"wrote {path_zip}: {len(out)} docs, {n_ent} entities")
    return out

# It takes s (random seed) as input, sets random seeds so runs are repeatable and returns nothing.
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# It takes gold (true labels) and pred (predictions) as input, counts the different kinds of mistakes and returns (mistake counts, mistakes per label).
def error_analysis(gold, pred):
    from collections import Counter
    cnt = Counter(); by = {l: Counter() for l in LABELS}
    for d in gold:
        g = [(int(a), int(b), l) for a, b, l, *_ in gold[d]]
        p = [(int(a), int(b), l) for a, b, l, *_ in pred.get(d, [])]
        gs, ps = set(g), set(p)
        for (s, e, l) in g:
            if (s, e, l) in ps: by[l]["correct"] += 1; continue
            ov = [q for q in p if q[0] < e and s < q[1]]
            if not ov: kind = "missed"
            elif any(q[0] == s and q[1] == e for q in ov): kind = "label"
            elif any(q[2] == l for q in ov): kind = "boundary"
            else: kind = "both"
            by[l][kind] += 1; cnt[kind] += 1
        for (s, e, l) in p:
            if (s, e, l) in gs: continue
            if not any(q[0] < e and s < q[1] for q in g):
                by[l]["spurious"] += 1; cnt["spurious"] += 1
    return cnt, by

# It takes cnt (mistake counts) and by (mistakes per label) as input, prints the mistake counts and returns nothing.
def print_errors(cnt, by):
    print("error types:", dict(cnt))
    print("%-10s %8s %8s %8s %8s %8s %9s" % ("gold label", "correct", "boundary", "label", "both", "missed", "spurious*"))
    for l in LABELS:
        b = by[l]
        print("%-10s %8d %8d %8d %8d %8d %9d" % (l, b["correct"], b["boundary"], b["label"], b["both"], b["missed"], b["spurious"]))
    print("(*spurious = predicted entities of that label that overlap no gold entity)")

# It takes path (file path), encs (encoded documents) and probs (probabilities) as input, saves predicted probabilities to a file and returns nothing.
def save_probs(path, encs, probs):
    np.savez_compressed(path, ids=np.array([e.id for e in encs]), **{f"p_{e.id}": p.astype(np.float16) for e, p in zip(encs, probs)})

# It takes repo (dataset name) and dest (download folder) as input, downloads the Phase-2 data if it is not attached and returns nothing.
def ensure_phase2_data(repo="Exploration-Lab/CS-781-Capstone", dest="/kaggle/working/phase2_data"):
    names = ["fold1-text.json", "fold1-gold.json", "fold2-text.json", "fold2-gold.json", "labels.txt", "fold3-text.json"]
    if all(find_file(n) for n in names[:5]):
        print("Phase-2 files found locally:", os.path.dirname(find_file("fold1-text.json")))
        return
    from huggingface_hub import HfApi, hf_hub_download, login
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("Hugging Face token (read access, dataset terms accepted): ")
    login(token=tok)
    files = HfApi().list_repo_files(repo, repo_type="dataset")
    os.makedirs(dest, exist_ok=True)
    for n in names:
        c = [f for f in files if f == n or f.endswith("/" + n)]
        pref = [f for f in c if any(k in f.lower() for k in ("ner", "phase_2", "phase2"))]
        c = pref or c
        if not c:
            if n == "fold3-text.json":
                print("  (fold3-text.json not found in the repo - test predictions will be skipped)"); continue
            raise RuntimeError(f"{n} not found in {repo}. Files look like: {files[:15]}")
        if len(c) > 1:
            print(f"  note: {n} appears {len(c)} times {c[:4]}; using {sorted(c, key=len)[0]}")
        p = hf_hub_download(repo, sorted(c, key=len)[0], repo_type="dataset", local_dir=dest)
        print("  downloaded", n, "->", p)

# It takes model, encs (encoded documents), W (window size), strides (window strides) and **kw (extra options) as input, averages predictions made with different window overlaps and returns averaged probabilities for every document.
def predict_probs_multi(model, encs, W, strides, **kw):
    acc = None
    for st in strides:
        p = predict_probs(model, encs, W, st, **kw)
        acc = p if acc is None else [a + b for a, b in zip(acc, p)]
    return [a / len(strides) for a in acc]

Writing ner_core.py


In [5]:
%%writefile ner_peft.py
# Fine-tuning methods: LoRA, LoRMA, adapters (saved as ner_peft.py)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# It takes model and name as input, gets a sub-layer of the model by its name and returns the sub-layer.
def _get(model, name):
    m = model
    for p in name.split("."):
        m = getattr(m, p)
    return m

# It takes model, name and new (new layer) as input, replaces a sub-layer of the model by its name and returns nothing.
def _set(model, name, new):
    parts = name.split(".")
    parent = _get(model, ".".join(parts[:-1])) if len(parts) > 1 else model
    setattr(parent, parts[-1], new)

class LoRALinear(nn.Module):
    # It takes base (base linear layer), r (rank), alpha (scale of the low-rank update) and dropout (dropout rate) as input, creates the small LoRA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the linear layer plus the LoRA update and returns the output of the updated layer.
    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))

class LoRMALinear(nn.Module):

    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.s = base, alpha / r
        o = base.out_features
        self.A = nn.Parameter(torch.empty(r, o)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(o, r))

    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.s * F.linear(F.linear(y0, self.A.to(x.dtype)), self.B.to(x.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(x.dtype)

# It takes model as input, freezes all weights of the model and returns nothing.
def freeze(model):
    for p in model.parameters():
        p.requires_grad_(False)

# It takes bb (pre-trained backbone), kind (type), names (layer names), r (rank), alpha (scale of the low-rank update) and dropout (dropout rate) as input, freezes the model and wraps chosen linear layers with LoRA or LoRMA and returns a number of layers wrapped.
def apply_linear_wrappers(bb, kind, names, r=8, alpha=16, dropout=0.05):
    freeze(bb.model)
    n = 0
    for name in names:
        base = _get(bb.model, name)
        if not isinstance(base, nn.Linear):
            continue
        new = LoRALinear(base, r, alpha, dropout) if kind == "lora" else LoRMALinear(base, r, alpha)
        _set(bb.model, name, new); n += 1
    assert n > 0, "no Linear layers wrapped - check the name list"
    return n

class Adapter(nn.Module):
    # It takes d (model size), bottleneck (adapter size) and dropout (dropout rate) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up, self.drop = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    # It takes h (hidden vectors) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(self, h):
        return h + self.up(self.drop(F.gelu(self.down(h))))

# It takes bb (pre-trained backbone) as input, returns the list of model blocks.
def get_blocks(bb):
    if hasattr(bb, "block_modules"):
        return list(bb.block_modules())
    return list(bb.model.blocks)

# It takes bb (pre-trained backbone), bottleneck (adapter size), dropout (dropout rate) and every (how often) as input, freezes the model and adds an adapter after the blocks and returns a number of adapters added.
def add_block_adapters(bb, bottleneck=64, dropout=0.1, every=1):
    freeze(bb.model)
    blocks = get_blocks(bb)
    n = 0
    for i, blk in enumerate(blocks):
        if i % every: continue
        blk.add_module("ner_adapter", Adapter(bb.d_model, bottleneck, dropout))

        # It takes mod (module), inp (layer input) and out (block output) as input, adds the adapter output to the block output and returns block output plus adapter output.
        def hook(mod, inp, out):
            ad = mod.ner_adapter
            if isinstance(out, tuple):
                return (ad(out[0]),) + tuple(out[1:])
            return ad(out)
        blk.register_forward_hook(hook); n += 1
    return n

# It takes bb (pre-trained backbone), last_k_blocks (number of top blocks to train), train_norms (also train norm layers) and train_embeddings (also train embeddings) as input, freezes the model except the last few blocks and returns a number of trainable parameters.
def partial_finetune(bb, last_k_blocks=4, train_norms=True, train_embeddings=False):
    freeze(bb.model)
    blocks = get_blocks(bb)
    for blk in blocks[-last_k_blocks:]:
        for p in blk.parameters(): p.requires_grad_(True)
    if train_norms:
        for n, p in bb.model.named_parameters():
            if "norm_f" in n or n.startswith("norm.") or n.startswith("ln_f"):
                p.requires_grad_(True)
    if train_embeddings:
        for n, p in bb.model.named_parameters():
            if "embed" in n or "tok" in n.split(".")[0]: p.requires_grad_(True)
    return sum(p.numel() for p in bb.model.parameters() if p.requires_grad)

# It takes bb (pre-trained backbone), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, gives lower layers a smaller learning rate than top layers and returns a list of parameter groups with learning rates.
def layerwise_lr_groups(bb, lr_top, decay=0.85, head_params=(), head_lr=1e-3, wd=0.01):
    blocks = get_blocks(bb)
    n = len(blocks)
    seen, groups = set(), []
    for i, blk in enumerate(blocks):
        ps = [p for p in blk.parameters() if p.requires_grad]
        for p in ps: seen.add(id(p))
        groups.append(dict(params=ps, lr=lr_top * decay ** (n - 1 - i), weight_decay=wd))
    rest = [p for p in bb.model.parameters() if p.requires_grad and id(p) not in seen]
    groups.append(dict(params=rest, lr=lr_top * decay ** n, weight_decay=wd))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return [g for g in groups if g["params"]]

Writing ner_peft.py


In [6]:
%%writefile ner_run.py
# Function to run one experiment (saved as ner_run.py)
import os, json, time, gc
import torch
import ner_core as C

# It frees unused CPU and GPU memory and returns nothing.
def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# It takes name, setup (function that builds the model), cfg (settings), encs (encoded documents), tr_ids (training document ids), va_ids (hold-out document ids), test_encs (encoded test documents), test_docs (test documents), out_dir (output folder), full_retrain (retrain on all data), device and soft_encs (documents with teacher probabilities) as input, runs one full experiment: train, evaluate and save outputs and returns a results summary.
def run_experiment(name, setup, cfg, encs, tr_ids, va_ids, test_encs, test_docs, out_dir, full_retrain=True, device="cuda", soft_encs=None):
    os.makedirs(out_dir, exist_ok=True)
    res = dict(name=name, cfg={k: v for k, v in cfg.items()})
    t0 = time.time()
    model, groups, info = setup(); model.to(device)
    tr_n, all_n = C.count_params(model); res.update(info=info, trainable_params=tr_n, total_params=all_n)
    print(f"[{name}] trainable {tr_n:,} of {all_n:,} parameters ({100*tr_n/all_n:.2f}%)")
    hist, best_ep, state = C.fit(model, [encs[k] for k in tr_ids], [encs[k] for k in va_ids], cfg, groups, device=device)
    C.load_best(model, state)
    sc, probs_va, pred_va = C.evaluate(model, [encs[k] for k in va_ids], cfg["W"], cfg["stride"], bs=max(2, cfg["bs"] * 2),
                                      double=cfg.get("double", False), pad_id=cfg.get("pad_id", 0), sep_id=cfg.get("sep_id", 1), device=device)
    C.print_scores(sc, f"[{name}] HOLD-OUT (best epoch {best_ep})")
    gold = {k: encs[k].spans for k in va_ids}
    cnt, by = C.error_analysis(gold, pred_va); C.print_errors(cnt, by)
    sc_arg = C.strict_scores(gold, C.probs_to_spans([encs[k] for k in va_ids], probs_va, "argmax"))
    print(f"[{name}] hold-out macro-F1 with plain argmax decoding: {sc_arg['macro_F1']:.4f} (Viterbi {sc['macro_F1']:.4f})")
    res.update(history=hist, best_epoch=best_ep, holdout=sc, holdout_argmax_macro_F1=sc_arg["macro_F1"],
               errors={"counts": dict(cnt), "by_label": {l: dict(by[l]) for l in C.LABELS}})
    C.save_probs(os.path.join(out_dir, f"{name}_holdout_probs.npz"), [encs[k] for k in va_ids], probs_va)
    soft = None
    if soft_encs:
        ps = C.predict_probs(model, soft_encs, cfg["W"], cfg["stride"], bs=max(2, cfg["bs"] * 2), double=cfg.get("double", False),
                             pad_id=cfg.get("pad_id", 0), sep_id=cfg.get("sep_id", 1), device=device)
        soft = {e.id: p for e, p in zip(soft_encs, ps)}
    files = []
    if test_encs:
        probs_te = C.predict_probs(model, test_encs, cfg["W"], cfg["stride"], bs=max(2, cfg["bs"] * 2), double=cfg.get("double", False),
                                   pad_id=cfg.get("pad_id", 0), sep_id=cfg.get("sep_id", 1), device=device)
        z = os.path.join(out_dir, f"{name}_dev_submission.zip")
        C.write_submission(C.probs_to_spans(test_encs, probs_te), test_docs, z); files.append(z)
        C.save_probs(os.path.join(out_dir, f"{name}_dev_test_probs.npz"), test_encs, probs_te)
    res["stage1_secs"] = time.time() - t0
    del model, groups, state; free()
    if full_retrain and test_encs:
        t1 = time.time()
        model, groups, _ = setup(); model.to(device)
        cfg2 = dict(cfg, epochs=max(1, best_ep), seed=cfg.get("seed", 0) + 1)
        C.fit(model, [encs[k] for k in list(tr_ids) + list(va_ids)], None, cfg2, groups, device=device)
        probs_te = C.predict_probs(model, test_encs, cfg["W"], cfg["stride"], bs=max(2, cfg["bs"] * 2), double=cfg.get("double", False),
                                   pad_id=cfg.get("pad_id", 0), sep_id=cfg.get("sep_id", 1), device=device)
        z = os.path.join(out_dir, f"{name}_full_submission.zip")
        C.write_submission(C.probs_to_spans(test_encs, probs_te), test_docs, z); files.append(z)
        C.save_probs(os.path.join(out_dir, f"{name}_full_test_probs.npz"), test_encs, probs_te)
        res["stage2_secs"] = time.time() - t1
        del model, groups; free()
    elif full_retrain:
        print(f"[{name}] no fold3-text.json found -> skipped the test predictions (attach the Phase-2 Test folder)")
    res["files"] = files
    json.dump(res, open(os.path.join(out_dir, f"{name}_report.json"), "w"), indent=1, default=float)
    if soft is not None: res["soft"] = soft
    return res

Writing ner_run.py


In [7]:
# Import the GraphProp model code
import arch_graphprop as A

In [8]:
# Load the NER data and make train / hold-out split
import sys, os, json, time, glob
import numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
import ner_core as C, ner_peft as P, ner_run as R
C.seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPUs:", torch.cuda.device_count())
os.makedirs(OUT, exist_ok=True)

if AUTO_DOWNLOAD: C.ensure_phase2_data()
docs = C.load_train_docs()
test_docs = C.load_test_docs()
if SMOKE:
    keep = sorted(docs)[:6]
    docs = {k: dict(docs[k], text=docs[k]["text"][:SMOKE_CHARS], spans=[s for s in docs[k]["spans"] if s[1] <= SMOKE_CHARS]) for k in keep}
    if test_docs: test_docs = {k: dict(v, text=v["text"][:SMOKE_CHARS]) for k, v in list(test_docs.items())[:2]}
    EPOCHS, N_VAL_DOCS, EVAL_FROM = 1, 2, 1
print(f"{len(docs)} labelled docs, test docs: {len(test_docs) if test_docs else 'NOT FOUND (attach Test/fold3-text.json)'}")

from collections import Counter
cnt = Counter(l for d in docs.values() for _, _, l in d["spans"])
nd = {l: sum(1 for d in docs.values() if any(s[2] == l for s in d["spans"])) for l in C.LABELS}
print("%-10s %7s %9s" % ("label", "entities", "in docs"))
for l in C.LABELS: print("%-10s %7d %6d/%d" % (l, cnt[l], nd[l], len(docs)))
orc = C.oracle_check(docs, list(docs))
print(f"\nORACLE: if the word tagger were perfect, strict macro-F1 would be {orc['macro_F1']:.4f}  (upper bound set by the word segmentation)")
print("labels the segmentation cannot reproduce exactly:", {l: round(orc['per_label'][l]['F1'], 3) for l in C.LABELS if orc['per_label'][l]['F1'] < 0.999})

tr_ids, va_ids = C.make_split(docs, n_val=N_VAL_DOCS, seed=SEED)
vc = Counter(l for k in va_ids for _, _, l in docs[k]["spans"])
print(f"\ndocument-level split: {len(tr_ids)} train docs / {len(va_ids)} hold-out docs; hold-out entities per label:", dict(vc))

device: cuda | GPUs: 2


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

  downloaded fold1-text.json -> /kaggle/working/phase2_data/Phase_2_LNER/Train/fold1-text.json


fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

  downloaded fold1-gold.json -> /kaggle/working/phase2_data/Phase_2_LNER/Train/fold1-gold.json


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

... (3 outputs left out)

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

  downloaded labels.txt -> /kaggle/working/phase2_data/Phase_2_LNER/Train/labels.txt


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

  downloaded fold3-text.json -> /kaggle/working/phase2_data/Phase_2_LNER/Test/fold3-text.json
70 labelled docs, test docs: 35
label      entities   in docs
APP            407     63/70
RESP           381     63/70
A.COUNSEL      195     49/70
R.COUNSEL      184     47/70
JUDGE          245     66/70
WIT            187     12/70
AUTH          1003     43/70
COURT         1013     66/70
STAT          1260     65/70
PREC          1183     66/70
DATE          1575     70/70
CASENO         760     65/70

ORACLE: if the word tagger were perfect, strict macro-F1 would be 0.9944  (upper bound set by the word segmentation)
labels the segmentation cannot reproduce exactly: {'APP': 0.985, 'RESP': 0.995, 'AUTH': 0.991, 'COURT': 0.982, 'STAT': 0.984, 'PREC': 0.995}

document-level split: 56 train docs / 14 hold-out docs; hold-out entities per label: {'COURT': 219, 'CASENO': 173, 'APP': 121, 'RESP': 78, 'STAT': 256, 'DATE': 329, 'AUTH': 209, 'JUDGE': 50, 'PREC': 295, 'A.COUNSEL': 45, 'R.COUNSEL': 48

In [9]:
# Build training config and learning-rate groups
# It takes bb (pre-trained backbone) and tok (tokenizer) as input, builds the training settings and returns a dictionary of training settings.
def make_cfg(bb, tok):
    W = min(WIN_TOKENS, (bb.max_len - 1) // 2 if DOUBLE_PASS else bb.max_len)
    return dict(epochs=EPOCHS, W=W, stride=min(STRIDE, max(1, W // 2)) if W < WIN_TOKENS else STRIDE, bs=MICRO_BS, accum=ACCUM, double=DOUBLE_PASS,
                rare_dup=RARE_DUP, class_power=CLASS_POWER, patience=PATIENCE, wd=WEIGHT_DECAY, seed=SEED, eval_from=EVAL_FROM, bs_state=dict(bs=MICRO_BS),
                pad_id=tok.pad_id if tok.pad_id is not None else 0, sep_id=tok.eos_id if tok.eos_id is not None else 1, amp=True)

# It takes model, bb_lr (backbone learning rate), head_lr (head learning rate) and wd (weight decay) as input, sets learning rates for the new head and the backbone and returns a list of parameter groups with learning rates.
def head_groups(model, bb_lr, head_lr=None, wd=WEIGHT_DECAY):
    head = [p for n, p in model.named_parameters() if not n.startswith("bb_model.") and p.requires_grad]
    bbp = [p for n, p in model.named_parameters() if n.startswith("bb_model.") and p.requires_grad]
    g = [dict(params=head, lr=head_lr or LR_HEAD, weight_decay=wd)]
    if bbp: g.append(dict(params=bbp, lr=bb_lr, weight_decay=wd))
    return g

In [10]:
# Load the pre-trained model and tokenize the documents
bb0 = A.load_backbone(CKPT, device="cpu")
tok = A.load_tokenizer(TOK)
print(f"backbone: {bb0.n_params/1e6:.1f}M parameters, d_model {bb0.d_model}, max_len {bb0.max_len} | tokenizer vocab {tok.vocab_size}")
CFG = make_cfg(bb0, tok); print("training config:", CFG)
encs = {e.id: e for e in C.build_encs(docs, sorted(docs), tok)}
print("tokenizer:", C.token_stats(list(encs.values())))
test_encs = C.build_encs(test_docs, sorted(test_docs), tok) if test_docs else None
del bb0
# It loads a fresh copy of the pre-trained model and returns the backbone.
def new_backbone():
    bb = A.load_backbone(CKPT, device="cpu")
    if BIDIRECTIONAL: bb.set_bidirectional(True)
    return bb

backbone: 210.2M parameters, d_model 1536, max_len 648 | tokenizer vocab 12000
training config: {'epochs': 36, 'W': 512, 'stride': 256, 'bs': 8, 'accum': 2, 'double': False, 'rare_dup': 1, 'class_power': 0.5, 'patience': 8, 'wd': 0.01, 'seed': 42, 'eval_from': 4, 'bs_state': {'bs': 8}, 'pad_id': 0, 'sep_id': 1, 'amp': True}
encoded 70 docs, 496,472 tokens, 449,428 words in 2s
tokenizer: {'tokens_per_char': 0.22435503025284828, 'chars_per_token': 4.457221212615555}
encoded 35 docs, 264,953 tokens, 239,512 words in 1s


In [11]:
# Set up the adapter and partial fine-tuning models
# It builds the model with adapters and returns (model, parameter groups, info).
def setup_adapter():
    bb = new_backbone()
    n = P.add_block_adapters(bb, ADAPTER_BOTTLENECK)
    model = C.NERModel(bb, GRU_HIDDEN, double=DOUBLE_PASS, sep_id=CFG["sep_id"])
    return model, head_groups(model, LR_ADAPTER), dict(technique="bottleneck adapters", n_adapters=n, bottleneck=ADAPTER_BOTTLENECK)

# It builds the model with partial fine-tuning and returns (model, parameter groups, info).
def setup_partial():
    bb = new_backbone()
    n = P.partial_finetune(bb, PARTIAL_K)
    model = C.NERModel(bb, GRU_HIDDEN, double=DOUBLE_PASS, sep_id=CFG["sep_id"])
    return model, head_groups(model, LR_PARTIAL), dict(technique="partial fine-tuning", last_k_blocks=PARTIAL_K, backbone_trainable=n)

## Run 1: adapters

In [12]:
# Train with adapters
if RUN_ADAPTER:

    res_adapter_graphprop = R.run_experiment("adapter_graphprop", setup_adapter, CFG, encs, tr_ids, va_ids, test_encs, test_docs, OUT, full_retrain=FULL_RETRAIN, device=DEVICE)

[adapter_graphprop] trainable 7,513,113 of 217,696,929 parameters (3.45%)
windows/epoch 2047, optimizer steps/epoch 127, total 4572, micro-batch 8 x accum 2


epoch 1/36:   0%|          | 0/127 [00:00<?, ?it/s]

epoch 1  train_ce 0.5631  secs 83.9285  | this run: epoch 1/36, ETA 0h48m  


epoch 2/36:   0%|          | 0/127 [00:00<?, ?it/s]

epoch 2  train_ce 0.0566  secs 82.9175  | this run: epoch 2/36, ETA 0h46m  


epoch 3/36:   0%|          | 0/127 [00:00<?, ?it/s]

... (39 outputs left out)

epoch 7/8:   0%|          | 0/165 [00:00<?, ?it/s]

epoch 7  train_ce 0.0019  secs 107.4761  | this run: epoch 7/8, ETA 0h01m  


epoch 8/8:   0%|          | 0/165 [00:00<?, ?it/s]

epoch 8  train_ce 0.0017  secs 107.9483  | this run: epoch 8/8, ETA 0h00m  
wrote /kaggle/working/adapter_graphprop_full_submission.zip: 35 docs, 4020 entities


## Run 2: partial fine-tuning

In [13]:
# Train with partial fine-tuning
if RUN_PARTIAL:

    res_partial_graphprop = R.run_experiment("partial_graphprop", setup_partial, CFG, encs, tr_ids, va_ids, test_encs, test_docs, OUT, full_retrain=FULL_RETRAIN, device=DEVICE)

[partial_graphprop] trainable 45,500,778 of 212,952,225 parameters (21.37%)
windows/epoch 2047, optimizer steps/epoch 127, total 4572, micro-batch 8 x accum 2


epoch 1/36:   0%|          | 0/127 [00:00<?, ?it/s]

epoch 1  train_ce 0.6871  secs 69.6084  | this run: epoch 1/36, ETA 0h40m  


epoch 2/36:   0%|          | 0/127 [00:00<?, ?it/s]

epoch 2  train_ce 0.0655  secs 70.3990  | this run: epoch 2/36, ETA 0h39m  


epoch 3/36:   0%|          | 0/127 [00:00<?, ?it/s]

... (67 outputs left out)

epoch 14/15:   0%|          | 0/165 [00:00<?, ?it/s]

epoch 14  train_ce 0.0016  secs 89.9389  | this run: epoch 14/15, ETA 0h01m  


epoch 15/15:   0%|          | 0/165 [00:00<?, ?it/s]

epoch 15  train_ce 0.0014  secs 90.7167  | this run: epoch 15/15, ETA 0h00m  
wrote /kaggle/working/partial_graphprop_full_submission.zip: 35 docs, 3755 entities


## Summary

In [14]:
# Show results table and plots
reports = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/*_report.json"))]
print("%-28s %8s %8s %8s %9s %10s" % ("experiment", "macroF1", "macroP", "macroR", "best ep", "trainable"))
for r in reports:
    h = r["holdout"]; print("%-28s %8.4f %8.4f %8.4f %9d %10s" % (r["name"], h["macro_F1"], h["macro_P"], h["macro_R"], r["best_epoch"], f"{r['trainable_params']:,}"))
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for r in reports:
    ax[0].plot([x["epoch"] for x in r["history"] if "val_macro_F1" in x], [x["val_macro_F1"] for x in r["history"] if "val_macro_F1" in x], marker="o", label=r["name"])
ax[0].set_title("hold-out strict macro-F1 per epoch"); ax[0].set_xlabel("epoch"); ax[0].legend()
w = 0.8 / max(1, len(reports))
for i, r in enumerate(reports):
    ax[1].bar(np.arange(12) + i * w, [r["holdout"]["per_label"][l]["F1"] for l in C.LABELS], w, label=r["name"])
ax[1].set_xticks(np.arange(12) + 0.4 - w / 2); ax[1].set_xticklabels(C.LABELS, rotation=45); ax[1].set_title("hold-out F1 per label"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/summary.png", dpi=110); plt.show()
print("files in", OUT); [print("  ", f) for f in sorted(os.listdir(OUT)) if f.endswith((".zip", ".json", ".npz", ".png"))]
print("\nSubmit the *_submission.zip files on Codabench (max 10 per day; start with the *_full_ ones - they were retrained on all 70 documents).")

experiment                    macroF1   macroP   macroR   best ep  trainable
adapter_graphprop              0.4730   0.4985   0.4606         8  7,513,113
partial_graphprop              0.4745   0.5132   0.4623        15 45,500,778
files in /kaggle/working
   adapter_graphprop_dev_submission.zip
   adapter_graphprop_dev_test_probs.npz
   adapter_graphprop_full_submission.zip
   adapter_graphprop_full_test_probs.npz
   adapter_graphprop_holdout_probs.npz
   adapter_graphprop_report.json
   partial_graphprop_dev_submission.zip
   partial_graphprop_dev_test_probs.npz
   partial_graphprop_full_submission.zip
   partial_graphprop_full_test_probs.npz
   partial_graphprop_holdout_probs.npz
   partial_graphprop_report.json
   summary.png

Submit the *_submission.zip files on Codabench (max 10 per day; start with the *_full_ ones - they were retrained on all 70 documents).
